# Novae Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 03.12.2024
- Date of Last Modification: 24.12.2025 (Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the novae conda environment, installable from ```../../envs/novae.yaml```
- This notebook follows the tutorial from https://mics-lab.github.io/novae/tutorials/main_usage/.

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import math
import os
import warnings

import anndata as ad
import pandas as pd
import novae
import numpy as np
import scanpy as sc
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'

### 1.3 Define Config

In [ ]:
min_cells_per_niche = 100

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/Novae', exist_ok=True)

## 2. Zero-shot Niche Identification

### 2.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

#### 2.1.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

#### 2.1.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
# This is not reproducible
model.compute_representations(adata, zero_shot=True, slide_key=batch_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded.h5ad')

In [ ]:
# Assign niches such that it matches ground truth with > 100 cells
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells - 12
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    'D1009',
    'D983',
    'D1001',
    'D998',
    'D970',
    'D986',
    'D1004',
    'D1011',
    'D994',
    'D989',
    'D1010',
    'D971',]

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_novae_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=f'novae_domains_{n_domains}',
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

In [ ]:
# Store on disk
metrics_df = pd.DataFrame({
    'n_niches': [n_domains],
    'NMI': [nmi_score],
    'ARI': [ari_score]
})
metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

#### 2.1.3 Quantitative Performance

No clustering required, therefore different resolutions are not tested.

### 2.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.2.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chr78-11b_1p'
batches = ['CM007',
           'CM008',
           'CM009',
           'CM011',
           'CM013',
           'CM015',
           'CM016',
           'CM017',
           'CM018',
           'CM019',
           'CM020']
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [240 * 0.093] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.093 # conversion rate from Valentina

In [ ]:
# Combine regions
adata.obs[niche_key] = adata.obs[niche_key].apply(lambda x: x.replace('Urogenital Sinus', 'UrogenitalSinus'))

#### 2.2.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
model.compute_representations(adata, zero_shot=True, slide_key=batch_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded.h5ad')

In [ ]:
# Assign niches such that it matches ground truth with > 100 cells
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
adata = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    'D998',
    'D1002',
    'D855',
    'D971',
    'D991',
    'D1000',
    'D1004',
    'D300',
    'D1003',
    'D997',
    'D983',
    'D892',
    'D215',
    'D993',
    'D828',
    'D856',
    'D999',
    'D994',
    'D956',]

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_novae_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=f'novae_domains_{n_domains}',
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

In [ ]:
# Store on disk
metrics_df = pd.DataFrame({
    'n_niches': [n_domains],
    'NMI': [nmi_score],
    'ARI': [ari_score]
})
metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

#### 2.2.3 Quantitative Performance

No clustering required, therefore different resolutions are not tested.

### 2.3 xhs49-1b_1p (Xenium Human Melanoma)

#### 2.3.1 Load Data

In [ ]:
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_size = 20

In [ ]:
adata = sc.read_h5ad(f'{st_data_silver_folder_path}/{dataset_name}/Xenium_Prime_Human_Skin_FFPE_outs.h5ad')

# Filter cells without niche labels
adata = adata[~adata.obs['niche_name'].isna()]

In [ ]:
adata.obs['niche_name'].value_counts()

#### 2.3.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
model.compute_representations(adata, zero_shot=True)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded.h5ad')

In [ ]:
# Assign niches such that it matches ground truth with > 100 cells
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
     'D1011',
     'D1016',
     'D1013',
     'D1007',
     'D995',
     'D1015',
     'D1014']

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_novae_niches.svg'
sc.pl.spatial(adata,
              color=latent_cluster_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

In [ ]:
# Store on disk
metrics_df = pd.DataFrame({
    'n_niches': [n_domains],
    'NMI': [nmi_score],
    'ARI': [ari_score]
})
metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

#### 2.3.3 Quantitative Performance

No clustering required, therefore different resolutions are not tested.

### 2.4 xhs1000-39b_1p (Xenium Human Skin - Eczema)

#### 2.4.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'xhs1000-39b_1p'
batches = ['batch11',
           'batch19',
           'batch32']
niche_key = 'niche_type'
batch_key = 'sample'
spot_sizes = [20] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial']

#### 2.4.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
# This is not reproducible
model.compute_representations(adata, zero_shot=True, slide_key=batch_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded.h5ad')

In [ ]:
# Assign niches such that it matches ground truth with > 100 cells
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells - 11
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    'D1007',
    'D1003',
    'D1010',
    'D765',
    'D963',
    'D1002',
    'D1012',
    'D974',
    'D1005',
    'D1011',
    'D1009',]

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_novae_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=f'novae_domains_{n_domains}',
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

In [ ]:
# Store on disk
metrics_df = pd.DataFrame({
    'n_niches': [n_domains],
    'NMI': [nmi_score],
    'ARI': [ari_score]
})
metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

#### 2.4.3 Quantitative Performance

No clustering required, therefore different resolutions are not tested.

## 3. Human Niche Benchmark (Shared Protocol, 2026)

- Added on 14.10.2026 by me13. Results of the human niche-identification benchmark, computed with the shared protocol in `../human_niche_benchmark/` (see its `README.md`).
- Same cells for every method (Section 2 loading); per-section spatial graphs; Leiden on a 15-NN graph of the embedding at the **lowest resolution giving the ground-truth number of niches** (niches with > 100 cells); NMI / ARI vs. ground truth.
- Computed by `../human_niche_benchmark/import_fm.py (import_fm.lsf, METHOD=novae) - imports the embeddings written by Section 2 of this notebook and scores them with score.py` on LSF; this section only reads the results from `artifacts/benchmarking/human/` and recomputes nothing.
- Novae follows the paper's mouse benchmark (`../mouse_benchmarking/novae_benchmarking.ipynb`): the native domains of Section 2 (`assign_domains` at the target level) are used for Novae's batch-effect correction, and Leiden is run on the corrected embedding with the shared protocol (resolution sweep, bars = sweep mean ± SD in the paper figure). The native-domain scores are saved as `*_novae_native_metric_scores.csv`.
- Runs in any environment with scanpy (e.g. the `terra (sb75)` kernel).

In [ ]:
import sys
sys.path.append('../human_niche_benchmark')
from method_report import show_method_metrics, show_method_maps

show_method_metrics('novae')

In [ ]:
# Cluster-to-niche matching and spatial maps (ground truth vs. method) per section
show_method_maps('novae')